# Insurance Cost Analysis

**Question:** what does a person's demographics and lifestyle tell us about their annual medical insurance charges — and which single factor matters most for pricing risk?

| Parameter | Description | Content type |
|---|----|---|
| age | Age in years | integer |
| gender | Male or Female | integer (1 or 2) |
| bmi | Body mass index | float |
| no_of_children | Number of children | integer |
| smoker | Whether smoker or not | integer (0 or 1) |
| region | Which US region — 0: northeast, 1: southeast, 2: southwest, 3: northwest | integer |
| charges | Annual medical insurance charges (target variable) | float |


## Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split

%matplotlib inline

## Load the Dataset

The raw CSV ships with no header row, so column names come later.


In [ ]:
file_name = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-Coursera/medical_insurance_dataset.csv'
df = pd.read_csv(file_name, header=None)
print(df.head(10))

## 1. Preparing the Raw Data

Add column names, then replace the `'?'` placeholder used for missing values with proper `NaN` so pandas' missing-data tools can find them.


In [ ]:
headers = ["age", "gender", "bmi", "no_of_children", "smoker", "region", "charges"]
df.columns = headers
df.replace('?', np.nan, inplace=True)
df.head()

## 2. Data Wrangling

### Which columns have missing data?


In [ ]:
print(df.info())

`age` and `smoker` are the columns with gaps. They call for different fixes: `age` is continuous, so the column mean is a reasonable stand-in; `smoker` is categorical (0/1), where a mean doesn't make sense — the most frequent value is the standard fallback instead.


In [ ]:
# smoker is a categorical attribute, replace missing values with the most frequent entry
is_smoker = df['smoker'].value_counts().idxmax()
df['smoker'].replace(np.nan, is_smoker, inplace=True)

# age is a continuous variable, replace missing values with the mean
mean_age = df['age'].astype('float').mean(axis=0)
df['age'].replace(np.nan, mean_age, inplace=True)

# update data types now that missing values are filled
df[['age', 'smoker']] = df[['age', 'smoker']].astype('int')

print(df.info())

Filling `smoker` with the most-frequent value means every imputed row gets labeled non-smoker — reasonable given non-smokers are the majority, but it's a real assumption: any imputed row that was actually a smoker will now understate that person's expected cost.


In [ ]:
df[['charges']] = np.round(df[['charges']], 2)
df.head()

## 3. Exploratory Data Analysis

### Does BMI predict cost?


In [ ]:
sns.regplot(x="bmi", y="charges", data=df, line_kws={"color": "red"})
plt.ylim(0,)

Expect a positive but fairly loose trend — BMI nudges charges upward on average, but the scatter should be wide. That's a sign one feature alone isn't telling the full story, which the boxplot below and the multi-feature model in Section 4 both speak to.


### Does smoking status predict cost?


In [ ]:
sns.boxplot(x="smoker", y="charges", data=df)

This is usually the single most dramatic split in the dataset — smokers' charges typically cluster in a much higher, wider range than non-smokers', with barely any overlap between the two boxes. If that holds here, it's a strong hint about which feature will dominate the models below.


### Full correlation matrix


In [ ]:
print(df.corr())

Check where `smoker` lands relative to `age` and `bmi` here — in most reproductions of this dataset it's by far the strongest correlate with `charges`, well ahead of every other column.


## 4. Model Development

### Baseline: smoking status alone


In [ ]:
X = df[['smoker']]
Y = df['charges']
lm = LinearRegression()
lm.fit(X, Y)
print(lm.score(X, Y))

A single yes/no feature explaining a large share of the variance in `charges` would be an unusually strong result for a one-variable model — worth calling out explicitly if it holds, since it's the kind of finding a stakeholder would want up front rather than buried in a correlation table.


### Does adding every attribute help?


In [ ]:
Z = df[['age', 'gender', 'bmi', 'no_of_children', 'smoker', 'region']]
lm.fit(Z, Y)
print(lm.score(Z, Y))

Compare this R² to the smoker-only baseline above. Expect an improvement, but likely a modest one relative to how much smoker status alone already explains — a useful data point for arguing that a pricing model needs more than one variable, even when one variable dominates.


### Adding non-linearity with a pipeline


In [ ]:
Input = [('scale', StandardScaler()), ('polynomial', PolynomialFeatures(include_bias=False)), ('model', LinearRegression())]
pipe = Pipeline(Input)
Z = Z.astype(float)
pipe.fit(Z, Y)
ypipe = pipe.predict(Z)
print(r2_score(Y, ypipe))

This is still an in-sample score, so treat any jump here cautiously — it may reflect the model fitting noise in the training data rather than a genuinely better fit. Section 5 tests that directly on held-out data.


## 5. Model Refinement

### Testing on unseen data

Split into training and test sets so the next two models are judged on customers they weren't fit on.


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(Z, Y, test_size=0.2, random_state=1)
print("number of test samples:", x_test.shape[0])
print("number of training samples:", x_train.shape[0])

In [ ]:
RidgeModel = Ridge(alpha=0.1)
RidgeModel.fit(x_train, y_train)
yhat = RidgeModel.predict(x_test)
print(r2_score(y_test, yhat))

Expect this test-set R² to sit close to (maybe slightly below) the in-sample full-feature score from Section 4 — a small gap is normal; a large one would mean that model was overfit.


### Combining regularization and non-linearity


In [ ]:
pr = PolynomialFeatures(degree=2)
x_train_pr = pr.fit_transform(x_train)
x_test_pr = pr.transform(x_test)

RidgeModel.fit(x_train_pr, y_train)
y_hat = RidgeModel.predict(x_test_pr)
print(r2_score(y_test, y_hat))

This is the real test of Section 4's polynomial pipeline: does it still win once it's scored on data it never saw during training? If this R² beats the plain Ridge model just above, the non-linear terms are adding real value; if not, the simpler model is the better one to recommend.


## Key Findings & Recommendation

*(Fill in with your actual R² values after running the notebook.)*

- **Smoking status alone** is typically the single strongest predictor of `charges` in this dataset — compare its solo R² (Section 4) against the full-feature model to see how much the other attributes really add on top of it.
- **Test-set comparison** (Section 5): does the polynomial + Ridge model beat the plain Ridge model on held-out data, or does the added complexity not pay off once it's judged on unseen customers?

**Recommendation:** insurers pricing risk from these attributes should weight smoking status most heavily, with age and BMI as secondary adjustors — and should validate any pricing model on held-out data (as done here) rather than in-sample fit alone.


---
*Dataset and task structure adapted from IBM's "Data Analysis with Python" course (Insurance Cost Analysis practice project). Analysis and code completed independently.*
